In [1]:
import sys
!{sys.executable} -m pip install opencv-python pandas numpy scikit-learn matplotlib seaborn


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:

import os
import pandas as pd
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns


# FONCTIONS D'ACTIVATION

def relu(x):
    """
    ReLU activation : max(0, x)
    """
  
    assert isinstance(x, np.ndarray), "Input to ReLU must be a numpy array"

    result = np.maximum(0, x)

    assert np.all(result >= 0), "ReLU output must be non-negative"
    return result


def relu_derivative(x):
    """
    Derivative of ReLU : 1 if x > 0, else 0
    """
    assert isinstance(x, np.ndarray), "Input to ReLU derivative must be a numpy array"

    result = (x > 0).astype(float)

    assert np.all((result == 0) | (result == 1)), "ReLU derivative must be 0 or 1"
    return result


def softmax(x):
    """
    Softmax activation : exp(x) / sum(exp(x))
    """
    assert isinstance(x, np.ndarray), "Input to softmax must be a numpy array"

    #  on soustrait le max de chaque ligne avant l'exponentielle
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))

    result = exp_x / np.sum(exp_x, axis=1, keepdims=True)
    assert np.all((result >= 0) & (result <= 1)), "Softmax output must be in [0, 1]"
    assert np.allclose(np.sum(result, axis=1), 1), "Softmax output must sum to 1 per sample"
    return result


# CLASSE : RESEAU DE NEURONES MULTICLASSE

class MultiClassNeuralNetwork:
    def __init__(self, layer_sizes, learning_rate=0.01):
        """
        Initialize the neural network with given layer sizes and learning rate.
        layer_sizes : List of integers [input_size, hidden1_size, ..., output_size]
        """
        assert isinstance(layer_sizes, list) and len(layer_sizes) >= 2, \
            "layer_sizes must be a list with at least 2 elements"
        assert all(isinstance(size, int) and size > 0 for size in layer_sizes), \
            "All layer sizes must be positive integers"
        assert isinstance(learning_rate, (int, float)) and learning_rate > 0, \
            "Learning rate must be a positive number"

        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        self.weights = []
        self.biases = []

        # Initialisation des poids et biais
        np.random.seed(42)
        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i], layer_sizes[i + 1]) * 0.01
            b = np.zeros((1, layer_sizes[i + 1]))
            assert w.shape == (layer_sizes[i], layer_sizes[i + 1]), \
                f"Weight matrix {i + 1} has incorrect shape"
            assert b.shape == (1, layer_sizes[i + 1]), \
                f"Bias vector {i + 1} has incorrect shape"
            self.weights.append(w)
            self.biases.append(b)

    def forward(self, X):
        """
        Forward propagation : Z^[l] = A^[l-1] W^[l] + b^[l], A^[l] = g(Z^[l])
        """
        assert isinstance(X, np.ndarray), "Input X must be a numpy array"
        assert X.shape[1] == self.layer_sizes[0], \
            f"Input dimension ({X.shape[1]}) must match input layer size ({self.layer_sizes[0]})"

        self.activations = [X]
        self.z_values = []

        #  boucle sur les couches CACHEES uniquement (toutes sauf la dernière)
        for i in range(len(self.weights) - 1):
            z = self.activations[-1] @ self.weights[i] + self.biases[i]
            assert z.shape == (X.shape[0], self.layer_sizes[i + 1]), \
                f"Z^[{i + 1}] has incorrect shape"
            self.z_values.append(z)
            self.activations.append(relu(z))   # [PROF] activation ReLU pour couches cachées

        #  même formule linéaire, mais pour la dernière couche
        z = self.activations[-1] @ self.weights[-1] + self.biases[-1]
        assert z.shape == (X.shape[0], self.layer_sizes[-1]), "Output Z has incorrect shape"
        self.z_values.append(z)

        #  activation softmax pour la couche de sortie 
        output = softmax(z)
        assert output.shape == (X.shape[0], self.layer_sizes[-1]), "Output A has incorrect shape"
        self.activations.append(output)

        return self.activations[-1]

    def compute_loss(self, y_true, y_pred):
        """
        Categorical Cross-Entropy : J = -1/m * sum(y_true * log(y_pred))
        """
        assert isinstance(y_true, np.ndarray) and isinstance(y_pred, np.ndarray), \
            "Inputs to loss must be numpy arrays"
        assert y_true.shape == y_pred.shape, "y_true and y_pred must have the same shape"

        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)  # [PROF] évite log(0)

        m = y_true.shape[0]
        loss = -np.sum(y_true * np.log(y_pred)) / m

        assert not np.isnan(loss), "Loss computation resulted in NaN"
        return loss

    def compute_accuracy(self, y_true, y_pred):
        """
        Compute accuracy : proportion of correct predictions
        """
        assert isinstance(y_true, np.ndarray) and isinstance(y_pred, np.ndarray), \
            "Inputs to accuracy must be numpy arrays"
        assert y_true.shape == y_pred.shape, "y_true and y_pred must have the same shape"

        # on compare la classe prédite (argmax) à la vraie classe (argmax du one-hot)
        predictions = np.argmax(y_pred, axis=1)
        true_labels = np.argmax(y_true, axis=1)
        accuracy = np.mean(predictions == true_labels)

        assert 0 <= accuracy <= 1, "Accuracy must be between 0 and 1"
        return accuracy

    def backward(self, X, y, outputs):
        """
        Backpropagation : compute dW^[l], db^[l] for each layer
        """
        assert isinstance(X, np.ndarray) and isinstance(y, np.ndarray) and isinstance(outputs, np.ndarray), \
            "Inputs to backward must be numpy arrays"
        assert X.shape[1] == self.layer_sizes[0], \
            f"Input dimension ({X.shape[1]}) must match input layer size ({self.layer_sizes[0]})"
        assert y.shape == outputs.shape, "y and outputs must have the same shape"

        m = X.shape[0]

        # on prépare des listes vides 
        # pour stocker les gradients de chaque couche
        self.d_weights = [np.zeros_like(w) for w in self.weights]
        self.d_biases = [np.zeros_like(b) for b in self.biases]

        # gradient de la couche de SORTIE 
        dZ = outputs - y  # Gradient pour softmax + cross-entropy
        assert dZ.shape == outputs.shape, "dZ for output layer has incorrect shape"
        self.d_weights[-1] = (self.activations[-2].T @ dZ) / m
        self.d_biases[-1] = np.sum(dZ, axis=0, keepdims=True) / m

        #  boucle en arrière sur les couches CACHEES
        for i in range(len(self.weights) - 2, -1, -1):
            dZ = (dZ @ self.weights[i + 1].T) * relu_derivative(self.z_values[i])
            assert dZ.shape == (X.shape[0], self.layer_sizes[i + 1]), \
                f"dZ^[{i + 1}] has incorrect shape"

            self.d_weights[i] = (self.activations[i].T @ dZ) / m
            self.d_biases[i] = np.sum(dZ, axis=0, keepdims=True) / m


        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * self.d_weights[i]
            self.biases[i] -= self.learning_rate * self.d_biases[i]

    def train(self, X, y, X_val, y_val, epochs, batch_size):
        """
        Train the neural network using mini-batch SGD, with validation
        """
        # toutes les assertions de vérification ---
        assert isinstance(X, np.ndarray) and isinstance(y, np.ndarray), "X and y must be numpy arrays"
        assert isinstance(X_val, np.ndarray) and isinstance(y_val, np.ndarray), \
            "X_val and y_val must be numpy arrays"
        assert X.shape[1] == self.layer_sizes[0], \
            f"Input dimension ({X.shape[1]}) must match input layer size ({self.layer_sizes[0]})"
        assert y.shape[1] == self.layer_sizes[-1], \
            f"Output dimension ({y.shape[1]}) must match output layer size ({self.layer_sizes[-1]})"
        assert X_val.shape[1] == self.layer_sizes[0], \
            f"Validation input dimension ({X_val.shape[1]}) must match input layer size ({self.layer_sizes[0]})"
        assert y_val.shape[1] == self.layer_sizes[-1], \
            f"Validation output dimension ({y_val.shape[1]}) must match output layer size ({self.layer_sizes[-1]})"
        assert isinstance(epochs, int) and epochs > 0, "Epochs must be a positive integer"
        assert isinstance(batch_size, int) and batch_size > 0, "Batch size must be a positive integer"

        train_losses = []
        val_losses = []
        train_accuracies = []
        val_accuracies = []

        for epoch in range(epochs):
            indices = np.random.permutation(X.shape[0])
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            epoch_loss = 0
            for i in range(0, X.shape[0], batch_size):
                X_batch = X_shuffled[i:i + batch_size]
                y_batch = y_shuffled[i:i + batch_size]

                outputs = self.forward(X_batch)
                epoch_loss += self.compute_loss(y_batch, outputs)
                self.backward(X_batch, y_batch, outputs)

            # Calculer les pertes et accuracies
            train_loss = epoch_loss / (X.shape[0] // batch_size)
            train_pred = self.forward(X)
            train_accuracy = self.compute_accuracy(y, train_pred)
            val_pred = self.forward(X_val)
            val_loss = self.compute_loss(y_val, val_pred)
            val_accuracy = self.compute_accuracy(y_val, val_pred)

            train_losses.append(train_loss)
            val_losses.append(val_loss)
            train_accuracies.append(train_accuracy)
            val_accuracies.append(val_accuracy)

            if epoch % 10 == 0:
                print(f"Epoch {epoch}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, "
                      f"Train Acc: {train_accuracy:.4f}, Val Acc: {val_accuracy:.4f}")

        return train_losses, val_losses, train_accuracies, val_accuracies

    def predict(self, X):
        """
        Predict class labels
        """
        assert isinstance(X, np.ndarray), "Input X must be a numpy array"
        assert X.shape[1] == self.layer_sizes[0], \
            f"Input dimension ({X.shape[1]}) must match input layer size ({self.layer_sizes[0]})"

        outputs = self.forward(X)
        predictions = np.argmax(outputs, axis=1)
        assert predictions.shape == (X.shape[0],), "Predictions have incorrect shape"
        return predictions


# CHARGEMENT ET PRETRAITEMENT DES DONNEES

ROOT_DATA_FOLDER = r"C:\Users\jook\.vscode\Desktop\master IAA\MASTER S2\deep_l\archive (1)\AMHCD_64\AMHCD_64"  # <-- CHANGE CETTE LIGNE avec ton propre chemin


def find_file(root_folder, filename):
    """[COMPLETE] Cherche un fichier par son nom, à n'importe quel niveau du dossier."""
    for dirpath, _, filenames in os.walk(root_folder):
        if filename in filenames:
            return os.path.join(dirpath, filename)
    return None


def find_folder(root_folder, folder_name_contains):
    """[COMPLETE] Cherche un sous-dossier dont le nom contient un mot-clé (ex: 'image')."""
    for dirpath, dirnames, _ in os.walk(root_folder):
        for d in dirnames:
            if folder_name_contains.lower() in d.lower():
                return os.path.join(dirpath, d)
    return None


print(f"Recherche des données dans : {ROOT_DATA_FOLDER}")

#  On cherche le fichier CSV des étiquettes,
csv_path = find_file(ROOT_DATA_FOLDER, "labels-map.csv")

# On cherche le dossier contenant les images 
images_dir = find_folder(ROOT_DATA_FOLDER, "image")
if images_dir is None:
    images_dir = ROOT_DATA_FOLDER  # repli : on suppose que les images sont à la racine

data_dir = images_dir
print(f"Dossier d'images trouvé : {data_dir}")
print(f"Fichier CSV trouvé : {csv_path}")

if csv_path is not None:
    #  Charger le fichier CSV contenant les étiquettes
    labels_df = pd.read_csv(csv_path)
    assert 'image_path' in labels_df.columns and 'label' in labels_df.columns, \
        "CSV must contain 'image_path' and 'label' columns"
    # Certains CSV donnent un chemin relatif au dossier CSV lui-même
    csv_folder = os.path.dirname(csv_path)
    labels_df['image_path'] = labels_df['image_path'].apply(
        lambda p: p if os.path.isabs(p) or os.path.exists(os.path.join(data_dir, p))
        else os.path.join(csv_folder, p)
    )
else:
    print("labels-map.csv introuvable. Construction du DataFrame à partir des sous-dossiers.")
    #  construire un DataFrame à partir des dossiers (un sous-dossier = une classe)
    image_paths = []
    labels = []
    for label_dir in os.listdir(data_dir):
        label_path = os.path.join(data_dir, label_dir)
        if os.path.isdir(label_path):
            for img_name in os.listdir(label_path):
                image_paths.append(os.path.join(label_path, img_name))
                labels.append(label_dir)
    labels_df = pd.DataFrame({'image_path': image_paths, 'label': labels})

#  Vérifier le DataFrame
assert not labels_df.empty, "No data loaded. Check dataset files."
print(f"Loaded {len(labels_df)} samples with {labels_df['label'].nunique()} unique classes.")

#  Encoder les étiquettes
label_encoder = LabelEncoder()
labels_df['label_encoded'] = label_encoder.fit_transform(labels_df['label'])
num_classes = len(label_encoder.classes_)


def load_and_preprocess_image(image_path, target_size=(32, 32)):
    """
    Load and preprocess an image : convert to grayscale, resize, normalize
    """
    assert os.path.exists(image_path), f"Image not found: {image_path}"
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    assert img is not None, f"Failed to load image: {image_path}"
    img = cv2.resize(img, target_size)
    img = img.astype(np.float32) / 255.0  # Normalisation
    return img.flatten()  # Aplatir pour le réseau de neurones


#  Charger toutes les images

def resolve_path(path):
    if os.path.exists(path):
        return path
    return os.path.join(data_dir, path)


X = np.array([load_and_preprocess_image(resolve_path(path))
              for path in labels_df['image_path']])
y = labels_df['label_encoded'].values

#  Vérifier les dimensions
assert X.shape[0] == y.shape[0], "Mismatch between number of images and labels"
assert X.shape[1] == 32 * 32, f"Expected flattened image size of {32*32}, got {X.shape[1]}"

# Diviser en ensembles d'entraînement, validation et test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42)

#  Convertir explicitement en NumPy arrays
X_train = np.array(X_train)
X_val = np.array(X_val)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

assert X_train.shape[0] + X_val.shape[0] + X_test.shape[0] == X.shape[0], \
    "Train-val-test split sizes must sum to total samples"

print(f"Train: {X_train.shape[0]} samples, Validation: {X_val.shape[0]} samples, Test: {X_test.shape[0]} samples")

# Encoder les étiquettes en one-hot pour la classification multiclasse
one_hot_encoder = OneHotEncoder(sparse_output=False)
y_train_one_hot = np.array(one_hot_encoder.fit_transform(y_train.reshape(-1, 1)))
y_val_one_hot = np.array(one_hot_encoder.transform(y_val.reshape(-1, 1)))
y_test_one_hot = np.array(one_hot_encoder.transform(y_test.reshape(-1, 1)))

assert isinstance(y_train_one_hot, np.ndarray), "y_train_one_hot must be a numpy array"
assert isinstance(y_val_one_hot, np.ndarray), "y_val_one_hot must be a numpy array"
assert isinstance(y_test_one_hot, np.ndarray), "y_test_one_hot must be a numpy array"

# CREATION ET ENTRAINEMENT DU MODELE

layer_sizes = [X_train.shape[1], 64, 32, num_classes]  # 64 et 32 neurones cachés, 33 classes
nn = MultiClassNeuralNetwork(layer_sizes, learning_rate=0.01)
train_losses, val_losses, train_accuracies, val_accuracies = nn.train(
    X_train, y_train_one_hot, X_val, y_val_one_hot, epochs=100, batch_size=32
)



# PREDICTIONS ET EVALUATION

y_pred = nn.predict(X_test)
print("\nRapport de classification (Test set):")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de confusion (Test set)')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.savefig('confusion_matrix.png')
plt.close()

# Courbes de perte et d'accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Courbe de perte
ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Validation Loss')
ax1.set_title('Courbe de perte')
ax1.set_xlabel('Époque')
ax1.set_ylabel('Perte')
ax1.legend()

# Courbe d'accuracy
ax2.plot(train_accuracies, label='Train Accuracy')
ax2.plot(val_accuracies, label='Validation Accuracy')
ax2.set_title('Courbe de précision')
ax2.set_xlabel('Époque')
ax2.set_ylabel('Précision')
ax2.legend()

plt.tight_layout()
fig.savefig('loss_accuracy_plot.png')
plt.close()

Recherche des données dans : C:\Users\jook\.vscode\Desktop\master IAA\MASTER S2\deep_l\archive (1)\AMHCD_64\AMHCD_64
Dossier d'images trouvé : C:\Users\jook\.vscode\Desktop\master IAA\MASTER S2\deep_l\archive (1)\AMHCD_64\AMHCD_64
Fichier CSV trouvé : None
labels-map.csv introuvable. Construction du DataFrame à partir des sous-dossiers.
Loaded 25740 samples with 33 unique classes.
Train: 15444 samples, Validation: 5148 samples, Test: 5148 samples
Epoch 0, Train Loss: 3.5039, Val Loss: 3.4965, Train Acc: 0.0304, Val Acc: 0.0305
Epoch 10, Train Loss: 3.5037, Val Loss: 3.4963, Train Acc: 0.0826, Val Acc: 0.0804
Epoch 20, Train Loss: 3.5003, Val Loss: 3.4918, Train Acc: 0.0476, Val Acc: 0.0478
Epoch 30, Train Loss: 2.8094, Val Loss: 2.7138, Train Acc: 0.1200, Val Acc: 0.1261
Epoch 40, Train Loss: 2.1373, Val Loss: 2.1962, Train Acc: 0.2615, Val Acc: 0.2570
Epoch 50, Train Loss: 1.5030, Val Loss: 1.3703, Train Acc: 0.5607, Val Acc: 0.5534
Epoch 60, Train Loss: 0.9424, Val Loss: 1.0057, Trai

In [3]:
import os
print(os.getcwd())

C:\Users\jook\tp_optimisation\tp3_deep
